<a href="https://colab.research.google.com/github/HariniMajety/Camera-DisplayPhoto/blob/main/Flight_Data_Spark_In_Class_Activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# if you run this notebook on your latop and you have already spark installed on your laptop,
# then you can remove this first cell.

# This cell installs pyspark on this collab session.

#!sudo apt update
#!apt-get install openjdk-21-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://downloads.apache.org/spark/spark-4.2.0/spark-4.2.0-bin-hadoop3.tgz
#!wget -q https://downloads.apache.org/spark/spark-4.2.0/spark-4.2.0-bin-hadoop3.tgz
#!tar xf spark-4.2.0-bin-hadoop3.tgz
!pip install -q findspark
!pip install pyspark
# !pip install py4j

import os
import sys
import findspark
findspark.init()
findspark.find()

import pyspark
from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("In-Class Spark Programming") \
       .getOrCreate()

spark

In [ ]:
from pyspark import SparkConf,SparkContext
sc = SparkContext.getOrCreate()

In [ ]:
# Download the files from Google Cloud
# !wget -q https://storage.googleapis.com/cs378/flights.csv.bz2
# !wget -q https://storage.googleapis.com/cs378/airports.csv.bz2

# large file is flights.csv.bz2
# !wget -q  https://www.cs.utexas.edu/~kiat/datasets/flights.csv.bz2

# https://www.cs.utexas.edu/~kiat/datasets/flights-10k.csv

!wget -q https://www.cs.utexas.edu/~kiat/datasets/flights.csv.bz2
!wget -q https://www.cs.utexas.edu/~kiat/datasets/airports.csv

In [ ]:
flights = sc.textFile("flights.csv.bz2")
airports = sc.textFile('airports.csv')

print('Header of Flights Data', flights.take(1))
print('Header of AIRPORT Data', airports.take(1))

Header of Flights Data ['YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY']
Header of AIRPORT Data ['IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE']


In [ ]:
# The data is about the flights from different airports which includes following attributes
#[u'YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY']


# Removing the Header Line from CSV file
linesHeader = flights.first()

flights_raw = flights\
    .zipWithIndex()\
    .filter(lambda x: x[1] > 1)\
    .keys()


flights = flights_raw\
    .map(lambda x: x.split(','))\
    .map(lambda x: (x[0], x[1], x[2], x[3], x[4],
                    x[5], x[6], x[7], x[8], x[9],
                    x[10], 0 if x[11]=='' else float(x[11]), x[12], x[13], x[14],
                    x[15], x[16], x[17], x[18], x[19],
                    x[20], x[21], x[22], x[23], x[24],
                    x[25], x[26], x[27], x[28], x[29], x[30]
))


airports_raw = airports\
    .zipWithIndex()\
    .filter(lambda x:x[1]>1)\
    .keys()

airports = airports_raw\
    .map(lambda x: x.split(','))\
    .map(lambda x: (x[0], x[1], x[2], x[3]))\
    .filter(lambda x: x[0])


flights_rdd = flights.map(lambda p: (p[0], p[1], p[2], p[3], p[4], p[5],
                                         p[6], p[7], p[8], p[9], p[10], p[11], p[24]))
# This is the following data items
# YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,
# CANCELLED

flights_rdd.take(1)
# YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY, CANCELLED


[('2015',
  '1',
  '1',
  '4',
  'AA',
  '2336',
  'N3KUAA',
  'LAX',
  'PBI',
  '0010',
  '0002',
  -8.0,
  '0')]

# Question-1: Get a list of all destination airports that one can fly from LAX airport with one direct flight.
# Index 7 is ORIGIN_AIRPORT
# index 8 is DESTINATION_AIRPORT

In [ ]:
# Add your code here
flights_rdd.filter(lambda x: x[7] == 'LAX').map(lambda x: x[8]).distinct().collect()

['PBI',
 'SLC',
 'BNA',
 'DAL',
 'ASE',
 'ELP',
 'MCI',
 'FLL',
 'MKE',
 'PIT',
 'ICT',
 'ITO',
 'MSN',
 'MSP',
 'IAH',
 'ORD',
 'PHX',
 'HNL',
 'IAD',
 'SMF',
 'MCO',
 'MRY',
 'MDW',
 'HOU',
 'SBA',
 'EUG',
 'CVG',
 'GEG',
 'ATL',
 'EWR',
 'STL',
 'PHL',
 'AUS',
 'CLD',
 'BZN',
 'COS',
 'LIH',
 'FAT',
 'SJC',
 'OAK',
 'SBP',
 'HDN',
 'OKC',
 'SFO',
 'PDX',
 'RDU',
 'DCA',
 'OGG',
 'BWI',
 'KOA',
 'RNO',
 'PSP',
 'CMH',
 'TPA',
 'MSY',
 'CLE',
 'MEM',
 'ANC',
 'MTJ',
 'OMA',
 'MIA',
 'CLT',
 'DEN',
 'DFW',
 'SEA',
 'JFK',
 'LAS',
 'BOS',
 'DTW',
 'SAN',
 'SAT',
 'TUS',
 'JAC',
 'BOI',
 'EGE',
 'ABQ',
 'SAF',
 'IND',
 'SMX',
 'RDM']

# Question-2: Which top 5 dates has the highst number of flights in a single day in this dataset?
# 0 YEAR, 1 MONTH, 2 DAY

In [ ]:
# Add your code here

date_cnt = flights_rdd.map(lambda x: ((x[0], x[1], x[2]), 1))\
                      .reduceByKey(lambda x, y: x + y)\
                      .top(5, key=lambda x: x[1])

print(date_cnt)

# Question-3: What is difference between average departure delay on Sundays (Day with id=6) in comparison with Wednesdays (Day with id=3)?


In [ ]:
# Add your code here


# Question-4: What are the top-3 Airplanes that are used a lot in different flights? Each airplane is uniquely identified by a single TAIL_NUMBER.

In [ ]:
# Add your code here


# Question 5:  What are the top 10 airport pairs (origin-destination) with the highest average departure delay (Only including flights that were not canceled)?



```
# This is formatted as code
```

# Question 6:  Which airline has the highest total air time across all its flights?



[link text](https://)

## Question 7: Which day of the week has the highest total number of delayed flights?

## Question 8: Which airline has the highest number of different planes? (Implement it with aggregate transmissions operations like aggregateByKey)

In [ ]:

def my_add(x, y):
    x.add(y)
    return x




# Question 9: Which top-3 cities have the largest departure dalys? You need to use both data sets and join them.  This task might take more time to finish. Implement an optimal solution here!

In [ ]:
#


# Question 10: To which city highest number of differnt Airlines are operating?